```markdown
# DeepLabCut 工具箱，使用 napari 进行标注
https://github.com/DeepLabCut/DeepLabCut

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1d409ffe-c9f4-47e1-bde2-3010c1c40455/naparidlc.png?format=500w)

本 Jupyter Notebook 演示了为您的项目使用 DeepLabCut 所需的必要步骤。
这里展示了最简单的实现代码，但许多函数都拥有额外的功能（参数和选项），因此请务必查阅[概述文档](<URL_PLACEHOLDER_1>)和协议论文！

本 Notebook 演示了如何执行以下操作：
- 创建一个项目
- 提取用于训练的图像帧
- 标注图像帧 [新增！使用 napari]
- 绘制已标注的图像
- 创建训练数据集
- 训练一个神经网络
- 评估网络性能
- 分析新的视频
- 创建一个自动标注的视频
- 绘制轨迹

本 Notebook 演示了为您的项目使用 DeepLabCut 所需的必要步骤。

这里展示了最简单的实现代码，但许多函数都拥有额外的功能，因此请务必查阅[概述文档](<URL_PLACEHOLDER_2>)和协议论文！

Nath\*, Mathis\* 等人：Using DeepLabCut for markerless pose estimation during behavior across species. Nature Protocols, 2019.

论文：https://www.nature.com/articles/s41596-019-0176-0

预印本：https://www.biorxiv.org/content/biorxiv/early/2018/11/24/476531.full.pdf
```

## 创建一个新项目

如果你希望使用不同的网络来分析数据，最好将这些项目分开。如果你正在跟踪相似的主题或条目（即使是在不同的环境中），你也应该使用一个项目。

此功能会在用户定义的目录中创建一个带有子目录和基本配置文件的新项目，否则项目将在当前工作目录中创建。

在项目的任何阶段，你都可以随时向项目中添加新的视频（用于标注更多数据）。

In [ ]:
import deeplabcut

In [ ]:
task = "Reaching" # Enter the name of your experiment Task
experimenter = "Mackenzie" # Enter the name of the experimenter
video = [
    "/Users/mwmathis/Documents/DeepLabCut/examples/Reaching-Mackenzie-2018-08-30/videos/reachingvideo1.avi"
] # Enter the paths of your videos OR FOLDER you want to grab frames from.

path_config_file = deeplabcut.create_new_project(task, experimenter, video, copy_videos=True) 

# NOTE: The function returns the path, where your project is.

# You could also enter this manually (e.g. if the project is already created and you 
# want to pick up, where you stopped...): Enter the path of the config file that was
# just created from the above step (check the folder)
# path_config_file = "/home/Mackenzie/Reaching/config.yaml"

## 现在，去编辑创建好的 `config.yaml` 文件吧！
添加你的身体部位标签，编辑每段视频需要提取的帧数，等等。

请注意，您可以通过在任何函数末尾添加一个 `?` 来查看更多关于它的信息，例如：

In [ ]:
deeplabcut.extract_frames?

## 从视频中提取帧

成功特征检测器的一个关键点是选择多样化的帧，这些帧应该能够代表你所研究的、需要被标注的行为。

此函数根据特定的视频（或文件夹），选择 $N$ 个均匀采样的帧（'uniform' 方式）。注意：如果目标行为是稀疏分布的，这种均匀采样可能无法产生足够多样化的帧（可以考虑使用 K-means 聚类），或者你也可以选择手动选择帧等其他方法。

另外，请确保从不同（行为）会话和不同动物中收集数据，如果这些因素差异很大，这样做有助于训练出具有不变性的特征检测器。

单个图像不应过大（即小于 $850 \times 850$ 像素）。虽然这个步骤可以稍后处理，但建议尽早裁剪帧，尽可能地移除帧中不必要的区域。

始终检查裁剪后的输出。如果对结果满意，则继续进行标注。

In [ ]:
# there are other ways to grab frames, such as uniformly; please see the paper:

# AUTOMATIC:
deeplabcut.extract_frames(path_config_file) 

## 标记提取的帧

只有配置文件中包含的视频才能用于提取帧。每条视频提取的标签都存储在项目目录下的 **'labeled-data'** 子目录中。每个子目录都以对应视频的名称命名。工具箱中包含一个可用于进行标记的标注工具箱。

In [ ]:
# Attention: If you have not installed the napari-dlc plugin, do so now by running this cell:
!pip install napari-deeplabcut

#if the plugin does not appear upon launch, consider running in the terminal the above command 
#within the same conda env and then re-starting kernel in your notebook (Kernel > restart).

In [ ]:
# napari will pop up! Please go to plugin > deeplabcut to start:
%gui qt6
import napari
napari.Viewer()

## 检查标签

[可选] 检查标签是否已正确创建并存储，对于训练是有益的，因为标注（Labeling）是创建训练数据集最关键的部分之一。DeepLabCut 工具箱提供了一个名为 `check_labels` 的函数来执行此操作。其用法如下：

In [ ]:
deeplabcut.check_labels(path_config_file) #this creates a subdirectory with the frames + your labels

如果需要调整这些标签，您可以使用重新启动标注的 GUI 来移动它们，保存，然后重新绘制（或绘图）！

## 创建训练数据集

此函数根据包含标签信息的 pandas DataFrame 生成网络训练所需的训练数据信息。用户可以在 `config.yaml` 文件中设置训练集大小的比例（基于 hd5 文件中所有已标记的图像）。在创建数据集时，如果用户想对性能进行基准测试（通常情况下，您只需要设置 1 份，所以可以不传入任何内容！），他们可以创建多个随机洗牌（shuffle）版本。

运行此脚本后，训练数据集将被创建并保存在项目目录下的 **`training-datasets`** 子目录中。

此函数还会**在 `dlc-models-pytorch` 下创建新的子目录**，并将项目的 `config.yaml` 文件更新，以包含指向正确的训练和测试姿态配置文件（pose configuration file）的路径。这些文件保存着训练网络的参数。工具箱中提供了一个示例配置文件，名为 **`pytorch_config.yaml`**。对于我们遇到的绝大多数使用场景，默认设置都是完全可以接受的。

现在，是时候开始训练网络了！

In [ ]:
deeplabcut.create_training_dataset(path_config_file)
#remember, there are several networks you can pick, the default is resnet-50!

## 开始训练：

此函数针对训练数据集的特定洗牌（随机打乱）进行网络的训练。

In [ ]:
deeplabcut.train_network(path_config_file)

## 开始评估
此函数用于评估在特定洗牌顺序（shuffle/shuffles）或所有洗牌顺序下，针对特定状态所训练的模型在数据集（图像）上的性能，并将结果作为 `.csv` 文件存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

**格式分析**: 输入文件是纯文本或 Markdown 格式（没有明显的 RST/MDX 结构，如 Frontmatter、指令或 JSX 标记）。将按 Markdown 规则处理。

## 开始分析视频

此函数用于分析新的视频。用户可以从评估结果中选择最佳模型，并在 `config.yaml` 文件中为变量 `snapshotindex` 指定正确的快照索引。否则，系统将默认使用最新的快照来分析视频。

分析结果将以 hd5 文件的形式存储在视频所在的同一目录下。

In [ ]:
videofile_path = ['videos/video3.avi','videos/video4.avi'] # Enter a folder OR a list of videos to analyze.

deeplabcut.analyze_videos(path_config_file,videofile_path, videotype='.avi')

## 提取异常帧 [可选步骤]

这是一个可选步骤，仅在评估结果很差（即标签预测错误）时使用。在这种情况下，用户可以使用以下函数来提取标签被错误预测的那些帧。此步骤有许多选项，请参阅：

In [ ]:
deeplabcut.extract_outlier_frames?

In [ ]:
deeplabcut.extract_outlier_frames(path_config_file,['/videos/video3.avi']) #pass a specific video

## 优化标签 [可选步骤]
在提取出异常帧之后，用户可以使用以下函数将预测的标签移动到正确的位置。从而增强训练数据集。

In [ ]:
#now you can edit the "machine-labeled file" within napari; 
#just again drop the file and images into the workspace after you load the plugin
%gui qt6
import napari
napari.Viewer()

**注意：** 之后，如果你想查看调整后的帧，可以通过运行以下命令在主 GUI 中加载它们：`deeplabcut.label_frames(path_config_file)`

（你可以在下方添加一个新的“单元格（cell）”来添加这段代码！）

#### 一旦所有文件夹都重新标记完毕，请再次检查标签！如果你不满意，请在主 GUI 中进行调整：

`deeplabcut.label_frames(path_config_file)`

检查标签：

`deeplabcut.check_labels(path_config_file)`

In [ ]:
#NOW, merge this with your original data:

deeplabcut.merge_datasets(path_config_file)

## 创建新一轮训练数据集 [可选步骤]

在完善了标签并将其追加到原始数据集之后，这就创建了新一轮的训练数据集。此设置已在 `config.yaml` 文件中自动配置完成，所以让我们开始训练吧！

In [ ]:
deeplabcut.create_training_dataset(path_config_file)

## 创建带标签视频

此函数用于可视化目的，可以用来创建一个 `.mp4` 格式的视频，其中包含网络预测的标签。该视频将保存在原始视频所在的同一目录中。

**此功能包含许多有趣（可配置）的选项！**

```python
deeplabcut.create_labeled_video(config, videos, videotype='avi', shuffle=1, trainingsetindex=0, filtered=False, save_frames=False, Frames2plot=None, delete=False, displayedbodyparts='all', codec='mp4v', outputframerate=None, destfolder=None, draw_skeleton=False, trailpoints=0, displaycropped=False)
```

因此，请务必检查：

In [ ]:
deeplabcut.create_labeled_video?

In [ ]:
deeplabcut.create_labeled_video(path_config_file,videofile_path)

## 绘制分析视频的轨迹
此函数将绘制整个视频中所有身体部位的轨迹。每个身体部位都由一个唯一的颜色来标识。

In [ ]:
%matplotlib notebook #for making interactive plots.
deeplabcut.plot_trajectories(path_config_file,videofile_path)